# 04 - Baseline Validation

Before adding ResNet50 or running the FFA ablation, we validate the
AlexNet baseline encoding model. This checks:

1. Shuffled-label control: does breaking the video-to-fMRI
   correspondence collapse accuracy to near zero?
2. Stability across seeds: how much does accuracy vary with different
   random train/test splits?
3. Alpha sensitivity: does the choice of ridge regularization strength
   change which layer looks best?
4. Feature standardization: does scaling activations before regression
   change the results?
5. Voxel-wise score distributions: is region-level accuracy driven by
   a few strong voxels or broadly consistent?

In [ ]:
!git clone https://github.com/sossyh/ffa-dnn-ablation.git
%cd ffa-dnn-ablation

In [ ]:
!pip install nilearn decord python-dotenv --quiet

In [ ]:
import os
from google.colab import userdata

os.environ["DROPBOX_DATASET_LINK"] = userdata.get("DROPBOX_DATASET_LINK")

In [ ]:
import sys
sys.path.append(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

from src.data_loading import download_algonauts_data, load_all_rois

In [ ]:
download_algonauts_data(data_dir="data/raw")

fmri_dir = "data/raw/participants_data_v2021/mini_track"
subject = "sub01"
roi_data = load_all_rois(fmri_dir, subject)

NUM_TRAIN_VIDEOS = roi_data["FFA"].shape[0]

activation_cache_path = "data/processed/alexnet_activations.npz"
cached = np.load(activation_cache_path, allow_pickle=True)
all_activations = cached["all_activations"].item()

for layer, acts in all_activations.items():
    assert acts.shape[0] == NUM_TRAIN_VIDEOS

print("Data loaded and shapes confirmed.")

## Reusable evaluation function

Same core logic as `src/encoding_model.py`, but with standardization as
an explicit option, since that's one of the things we're checking.

In [ ]:
def evaluate(X, Y, alpha=100.0, n_folds=5, seed=42, standardize=True):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    num_voxels = Y.shape[1]
    fold_scores = np.zeros((n_folds, num_voxels))

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X)):
        X_train, X_test = X[train_idx], X[test_idx]
        Y_train, Y_test = Y[train_idx], Y[test_idx]

        if standardize:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        model = Ridge(alpha=alpha)
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)

        for v in range(num_voxels):
            r, _ = pearsonr(Y_test[:, v], Y_pred[:, v])
            fold_scores[fold_idx, v] = r if not np.isnan(r) else 0.0

    return fold_scores.mean(axis=0)

## Check 1: Shuffled-label control

We randomly shuffle which fMRI row goes with which video, breaking the
true correspondence, then re-run the regression. If real accuracy is
genuine signal, shuffled accuracy should collapse to near zero.

In [ ]:
X = all_activations["fc7"]
Y = roi_data["FFA"]

real_scores = evaluate(X, Y)
print("Real (correctly paired) mean accuracy:", real_scores.mean())

rng = np.random.default_rng(seed=0)
shuffled_idx = rng.permutation(Y.shape[0])
Y_shuffled = Y[shuffled_idx]

shuffled_scores = evaluate(X, Y_shuffled)
print("Shuffled (broken pairing) mean accuracy:", shuffled_scores.mean())

print("\nIf the shuffled accuracy is close to zero and much lower than")
print("the real accuracy, this confirms the model is learning genuine")
print("structure, not an artifact.")

## Check 2: Stability across seeds

Re-run the same fc7 -> FFA regression with several different random
seeds for the train/test split, and look at how much the mean accuracy
varies.

In [ ]:
seeds = [0, 1, 2, 3, 4, 42, 100]
seed_results = []

for seed in seeds:
    scores = evaluate(X, Y, seed=seed)
    seed_results.append({"seed": seed, "mean_accuracy": scores.mean()})
    print(f"seed={seed}: mean accuracy = {scores.mean():.4f}")

seed_df = pd.DataFrame(seed_results)
print("\nStd across seeds:", seed_df["mean_accuracy"].std())
print("A small std relative to the mean accuracy indicates stable results.")

## Check 3: Alpha sensitivity

Sweep alpha across several orders of magnitude for a few key layer /
region pairs, and see whether the ranking of layers (e.g. fc7 being
best) holds up regardless of the exact alpha chosen.

In [ ]:
alphas = [1, 10, 100, 1000, 10000]
layers_to_check = ["conv5", "fc6", "fc7", "fc8"]

alpha_results = []

for layer in layers_to_check:
    X_layer = all_activations[layer]
    for alpha in alphas:
        scores = evaluate(X_layer, Y, alpha=alpha)
        alpha_results.append({
            "layer": layer,
            "alpha": alpha,
            "mean_accuracy": scores.mean(),
        })
        print(f"{layer}, alpha={alpha}: {scores.mean():.4f}")

alpha_df = pd.DataFrame(alpha_results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for layer in layers_to_check:
    subset = alpha_df[alpha_df["layer"] == layer]
    ax.plot(subset["alpha"], subset["mean_accuracy"], marker="o", label=layer)

ax.set_xscale("log")
ax.set_xlabel("Alpha (log scale)")
ax.set_ylabel("Mean FFA accuracy")
ax.set_title("Alpha sensitivity across layers (FFA)")
ax.legend()
fig.tight_layout()

os.makedirs("results/figures/alexnet", exist_ok=True)
fig.savefig("results/figures/alexnet/alpha_sensitivity.png", dpi=150)
plt.show()

print("\nIf fc7 stays on top (or close to it) across most alpha values,")
print("the conclusion that fc7 is the best layer for FFA is not just an")
print("artifact of the specific alpha=100.0 default.")

## Check 4: Standardization effect

Compare accuracy with and without feature standardization, for the
same layer/region pair.

In [ ]:
scores_with_std = evaluate(X, Y, standardize=True)
scores_without_std = evaluate(X, Y, standardize=False)

print("With standardization:", scores_with_std.mean())
print("Without standardization:", scores_without_std.mean())
print("\nA large difference here means standardization matters and should")
print("be applied consistently across every model/layer for fair comparison.")

## Check 5: Voxel-wise score distribution

Instead of just the mean, look at the full distribution of individual
voxel accuracy scores within FFA, to see whether the region-level
average is driven by a few strong voxels or broadly consistent.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(real_scores, bins=20, edgecolor="black")
ax.set_xlabel("Per-voxel correlation (accuracy)")
ax.set_ylabel("Number of voxels")
ax.set_title("Distribution of voxel-wise accuracy scores (FFA, fc7)")
fig.tight_layout()
fig.savefig("results/figures/alexnet/ffa_voxel_score_distribution.png", dpi=150)
plt.show()

print("Fraction of voxels with accuracy > 0.1:",
      (real_scores > 0.1).mean())
print("Fraction of voxels with accuracy > 0.2:",
      (real_scores > 0.2).mean())

## Summary and download

In [ ]:
os.makedirs("results/tables/alexnet", exist_ok=True)

seed_df.to_csv("results/tables/alexnet/validation_seed_stability.csv", index=False)
alpha_df.to_csv("results/tables/alexnet/validation_alpha_sweep.csv", index=False)

summary = pd.DataFrame([{
    "real_accuracy": real_scores.mean(),
    "shuffled_accuracy": shuffled_scores.mean(),
    "seed_std": seed_df["mean_accuracy"].std(),
    "with_standardization": scores_with_std.mean(),
    "without_standardization": scores_without_std.mean(),
}])
summary.to_csv("results/tables/alexnet/validation_summary.csv", index=False)
print(summary)

In [ ]:
from google.colab import files

files.download("results/tables/alexnet/validation_seed_stability.csv")
files.download("results/tables/alexnet/validation_alpha_sweep.csv")
files.download("results/tables/alexnet/validation_summary.csv")
files.download("results/figures/alexnet/alpha_sensitivity.png")
files.download("results/figures/alexnet/ffa_voxel_score_distribution.png")

## Interpreting the results

Before moving to ResNet50 or the FFA ablation, check:

- Shuffled accuracy should be close to 0 and clearly below real accuracy.
- Seed-to-seed variation (std) should be small relative to the mean
  accuracy.
- fc7 should remain at or near the top across most alpha values in the
  sweep plot.
- Standardization's effect should be noted either way, and applied
  consistently going forward.
- The voxel-wise histogram shows whether FFA's accuracy is broadly
  distributed or concentrated in a few voxels.

If all of these look reasonable, the baseline is validated and it's
safe to move on to `notebooks/02b_baseline_encoding_model_any_model.ipynb`
for ResNet50, and then the FFA ablation.